In [ ]:
import cv2
import json
import os
import numpy as np
from tqdm import tqdm

# 🔥 USE FULL PATH (avoid wrong file issue)
file_path_1 = r"C:\Users\sonal\Projectpr\Dictionary.json"

output_folder = "Preprocessed_datasets"
dataset_name = "PlantVillage"

def load_data(dataset, file_path, output_folder):

    input_shape = (224, 224, 3)

    # ✅ load correct JSON
    print("Loading file:", file_path)
    with open(file_path, 'r') as d:
        dic = json.load(d)

    dic = dic[dataset]

    dataset_output_folder = os.path.join(output_folder, dataset)
    os.makedirs(dataset_output_folder, exist_ok=True)

    # 🔥 FIXED LOOP (IMPORTANT)
    for category in dic:   # color / grayscale / segmented
        print(f"\n📂 Category: {category}")

        for disease in dic[category]:
            for plant in dic[category][disease]:

                image_paths = dic[category][disease][plant]

                for img_path in tqdm(image_paths,
                                     desc=f"{category}-{disease}-{plant}"):

                    image = cv2.imread(img_path)

                    if image is None:
                        continue

                    h, w = image.shape[:2]

                    # 🔥 Make square
                    if w != h:
                        if w < h:
                            x = (h - w) // 2
                            image = cv2.copyMakeBorder(
                                image, 0, 0, x, x,
                                cv2.BORDER_CONSTANT,
                                value=(255, 255, 255)
                            )
                        else:
                            x = (w - h) // 2
                            image = cv2.copyMakeBorder(
                                image, x, x, 0, 0,
                                cv2.BORDER_CONSTANT,
                                value=(255, 255, 255)
                            )

                    # 🔥 Resize
                    resized_image = cv2.resize(
                        image, (224, 224),
                        interpolation=cv2.INTER_LANCZOS4
                    )

                    # 🔥 Normalize
                    normalized_image = resized_image.astype(np.float32) / 255.0

                    base_name = os.path.splitext(
                        os.path.basename(img_path)
                    )[0]

                    # 🔥 Save path (category included)
                    output_path = os.path.join(
                        dataset_output_folder,
                        category,
                        disease,
                        plant
                    )

                    os.makedirs(output_path, exist_ok=True)

                    output_file = os.path.join(
                        output_path,
                        f"{base_name}.npz"
                    )

                    np.savez_compressed(
                        output_file,
                        normalized_image=normalized_image
                    )

    print("\n✅ Preprocessing Done!")


# 🚀 RUN
load_data(dataset_name, file_path_1, output_folder)

Spider_mites Two-spotted_spider_mite-Tomato: 100%|██████████| 1676/1676 [01:35<00:00, 17.54it/s]
